# Subject 01 Visual Brain VAE Bottleneck + CLIP MLP
Train a reconstruction-only fMRI VAE, freeze it, then use a supervised MLP to translate its bottleneck embedding to CLIP image embeddings.


# 1. Train + Eval fMRI Autoencoder
Load fMRI responses, train the VAE, and inspect reconstruction quality.


## Setup
Mount Drive, import libraries, set paths, and define compact hyperparameters.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, random, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.io import loadmat
import urllib.request

BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/MyDrive/maxwell-braid"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
BETA_DIR = f"{BASE}/subject01_visual_brain_responses"
CLIP_DIR = f"{BASE}/subject01_clip_image_embeddings"
FULL_IMAGE_CACHE_PATH = f"{INPUT_DIR}/nsd_all_images_u8.pt"  # Optional tensor indexed by NSD image id.
SHARED1000_CACHE_PATH = f"{INPUT_DIR}/shared1000_ground_truth_images_u8.pt"
LOCAL_SHARED1000_CACHE_PATH = f"{BASE}/shared1000_ground_truth_images_u8.pt"
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
# TIMESTAMP = "20260823_190511"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs/vae_bottleneck_clip_mlp_{TIMESTAMP}"
SUBJECT = "subj01"

TEST_EVAL_N = 1000  # Shared-1000 eval rows; set lower for faster smoke tests.

SEED = 0
VAL_FRAC = 0.10
TEST_FRAC = 0.10
BATCH_SIZE = 512
VAE_EPOCHS = 50
VAE_LR = 1e-3
VAE_WEIGHT_DECAY = 1e-4
VAE_HIDDEN_DIM = 1024
LATENT_DIM = 1280
VAE_DROPOUT = 0.10
VAE_BETA = 1e-3
CLIP_MLP_EPOCHS = 75
CLIP_MLP_LR = 3e-4
CLIP_MLP_HIDDEN_DIM = 1024
CLIP_MLP_DROPOUT = 0.10
CLIP_MLP_MSE_WEIGHT = 1.0
CLIP_MLP_COS_WEIGHT = 1.0
CLIP_MLP_CONTRASTIVE_WEIGHT = 0.1
CLIP_MLP_TEMPERATURE = 0.07
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name in ["subject01_visual_brain_responses", "subject01_clip_image_embeddings"]:
    dst = f"{BASE}/{name}"
    if not os.path.exists(dst):
        shutil.copytree(f"{INPUT_DIR}/{name}", dst)

print(f"device={DEVICE} output={OUTPUT_DIR}")


## Load Data
Load matched fMRI and CLIP session tensors, then split into train, validation, and test.


In [ ]:
def session_id(path):
    return int(re.findall(r"\d+", os.path.basename(path))[-1])

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
clip_sess = {session_id(p): p for p in glob.glob(f"{CLIP_DIR}/{SUBJECT}_clip_embeds*.pt")}
sessions = sorted(set(beta_sess) & set(clip_sess))
assert sessions, "No matched fMRI/CLIP sessions found."

betas = torch.cat([torch.load(beta_sess[s], map_location="cpu").float() for s in tqdm(sessions, desc="load fMRI")])
clips = torch.cat([torch.load(clip_sess[s], map_location="cpu").float() for s in tqdm(sessions, desc="load CLIP")])
assert len(betas) == len(clips), (betas.shape, clips.shape)

EXP = f"{BASE}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)

mat = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
sharedix = mat["sharedix"].reshape(-1).astype(np.int64) - 1
imgbrick_ids = subjectim[int(SUBJECT[-2:]) - 1, masterordering]

trial_image_ids = []
for gidx in range(len(betas)):
    session = sessions[int(gidx) // 750]
    offset = int(gidx) % 750
    trial_image_ids.append(int(imgbrick_ids[(session - 1) * 750 + offset]))
trial_image_ids = np.array(trial_image_ids, dtype=np.int64)

image_to_trial_idxs = {}
for idx, image_id in enumerate(trial_image_ids):
    image_to_trial_idxs.setdefault(int(image_id), []).append(idx)

rng = np.random.RandomState(SEED)
unique_image_ids = np.array(sorted(image_to_trial_idxs), dtype=np.int64)
shared_eval_image_ids = np.array([int(image_id) for image_id in sharedix if int(image_id) in image_to_trial_idxs], dtype=np.int64)
if len(shared_eval_image_ids) != len(sharedix):
    print(f"warning: subject data contains {len(shared_eval_image_ids)} of {len(sharedix)} shared images")

nonshared_image_ids = np.array([image_id for image_id in unique_image_ids if image_id not in set(shared_eval_image_ids.tolist())], dtype=np.int64)
shuffled_nonshared_image_ids = rng.permutation(nonshared_image_ids)
n_val_images = max(1, int(VAL_FRAC * len(shuffled_nonshared_image_ids)))
val_image_ids = shuffled_nonshared_image_ids[:n_val_images]
train_image_ids = shuffled_nonshared_image_ids[n_val_images:]
test_image_ids = shared_eval_image_ids

def trial_idxs_for_images(image_ids):
    return torch.tensor([idx for image_id in image_ids for idx in image_to_trial_idxs[int(image_id)]], dtype=torch.long)

train_idx = trial_idxs_for_images(train_image_ids)
val_idx = trial_idxs_for_images(val_image_ids)
test_idx = trial_idxs_for_images(test_image_ids)

# Image-level eval rows: one averaged fMRI response + one paired CLIP embedding per shared-1000 image.
def aggregate_image_rows(image_ids, desc):
    xs, ys, ids = [], [], []
    for image_id in tqdm(list(map(int, image_ids)), desc=desc):
        idxs = torch.tensor(image_to_trial_idxs[image_id], dtype=torch.long)
        xs.append(betas[idxs].mean(0))
        ys.append(clips[idxs[0]])
        ids.append(image_id)
    return torch.stack(xs), torch.stack(ys), np.array(ids, dtype=np.int64)

test_eval_betas, test_eval_clips, test_eval_image_ids = aggregate_image_rows(shared_eval_image_ids, "aggregate shared-1000 eval images")
if TEST_EVAL_N is not None and len(test_eval_image_ids) > int(TEST_EVAL_N):
    keep = np.arange(int(TEST_EVAL_N))
    test_eval_betas = test_eval_betas[keep]
    test_eval_clips = test_eval_clips[keep]
    test_eval_image_ids = test_eval_image_ids[keep]

beta_mean = betas[train_idx].mean(0)
beta_std = betas[train_idx].std(0).clamp_min(1e-6)
clip_mean = clips[train_idx].mean(0)
clip_std = clips[train_idx].std(0).clamp_min(1e-6)
x = (betas - beta_mean) / beta_std
y_clip = (clips - clip_mean) / clip_std
x_test_eval = (test_eval_betas - beta_mean) / beta_std
y_clip_test_eval = (test_eval_clips - clip_mean) / clip_std
INPUT_DIM = x.shape[1]
CLIP_DIM = y_clip.shape[1]

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(TensorDataset(x[train_idx]), batch_size=BATCH_SIZE, shuffle=True, generator=g, pin_memory=True)
train_eval_loader = DataLoader(TensorDataset(x[train_idx]), batch_size=BATCH_SIZE, pin_memory=True)
val_loader = DataLoader(TensorDataset(x[val_idx]), batch_size=BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(TensorDataset(x_test_eval), batch_size=BATCH_SIZE, pin_memory=True)

print(f"sessions={sessions}")
print(f"betas={tuple(betas.shape)} clips={tuple(clips.shape)}")
print(f"train_images={len(train_image_ids)} val_images={len(val_image_ids)} shared_test_images={len(test_image_ids)} eval_images={len(test_eval_image_ids)}")
print(f"train_trials={len(train_idx)} val_trials={len(val_idx)} test_trials={len(test_idx)}")


## VAE Architecture
Define the fMRI variational autoencoder and its reconstruction plus KL loss.


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, dropout):
        super().__init__()
        mid = hidden_dim // 2
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, mid), nn.ReLU(),
        )
        self.mu = nn.Linear(mid, latent_dim)
        self.logvar = nn.Linear(mid, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, mid), nn.ReLU(),
            nn.Linear(mid, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        return self.decoder(self.reparameterize(mu, logvar)), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    recon = F.mse_loss(recon_x, x)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + VAE_BETA * kl, recon, kl

model = VAE(INPUT_DIM, VAE_HIDDEN_DIM, LATENT_DIM, VAE_DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=VAE_LR, weight_decay=VAE_WEIGHT_DECAY)


## Train VAE
Train the fMRI autoencoder and save the checkpoint with the best validation loss.


In [ ]:
@torch.no_grad()
def evaluate_vae(loader):
    model.eval()
    totals = np.zeros(3)
    seen = 0
    for (xb,) in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        recon_x, mu, logvar = model(xb)
        loss, recon, kl = vae_loss(recon_x, xb, mu, logvar)
        totals += xb.size(0) * np.array([loss.item(), recon.item(), kl.item()])
        seen += xb.size(0)
    return dict(zip(["loss", "recon", "kl"], totals / seen))

vae_history = []
best_val = float("inf")
for epoch in range(1, VAE_EPOCHS + 1):
    model.train()
    train_totals = np.zeros(3)
    seen = 0
    pbar = tqdm(train_loader, desc=f"vae {epoch:03d}/{VAE_EPOCHS}", leave=False)
    for (xb,) in pbar:
        xb = xb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        recon_x, mu, logvar = model(xb)
        loss, recon, kl = vae_loss(recon_x, xb, mu, logvar)
        loss.backward()
        optimizer.step()
        train_totals += xb.size(0) * np.array([loss.item(), recon.item(), kl.item()])
        seen += xb.size(0)
        pbar.set_postfix(loss=loss.item(), recon=recon.item(), kl=kl.item())

    row = {f"train_{k}": v for k, v in zip(["loss", "recon", "kl"], train_totals / seen)}
    row.update({f"val_{k}": v for k, v in evaluate_vae(val_loader).items()})
    row["epoch"] = epoch
    vae_history.append(row)
    tqdm.write(f"vae_epoch={epoch:03d} train={row['train_loss']:.4f} val={row['val_loss']:.4f}")

    if row["val_loss"] < best_val:
        best_val = row["val_loss"]
        torch.save({"model": model.state_dict(), "beta_mean": beta_mean, "beta_std": beta_std}, f"{OUTPUT_DIR}/best_vae.pt")

with open(f"{OUTPUT_DIR}/vae_history.json", "w") as f:
    json.dump(vae_history, f, indent=2)


## Evaluate VAE
Reload the best VAE, report final split losses, and plot train vs validation loss.


In [ ]:
ckpt = torch.load(f"{OUTPUT_DIR}/best_vae.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
final_train = evaluate_vae(train_eval_loader)
final_val = evaluate_vae(val_loader)
final_test = evaluate_vae(test_loader)
print(f"final_train_loss={final_train['loss']:.6f}")
print(f"final_val_loss={final_val['loss']:.6f}")
print(f"final_test_loss={final_test['loss']:.6f}")
print(f"train_recon={final_train['recon']:.6f} val_recon={final_val['recon']:.6f} test_recon={final_test['recon']:.6f}")
print(f"train_kl={final_train['kl']:.6f} val_kl={final_val['kl']:.6f} test_kl={final_test['kl']:.6f}")

epochs = [h["epoch"] for h in vae_history]
plt.plot(epochs, [h["train_loss"] for h in vae_history], label="train")
plt.plot(epochs, [h["val_loss"] for h in vae_history], label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()


# 2. Train + Eval Bottleneck CLIP MLP
Freeze the fMRI autoencoder and train/evaluate a supervised MLP from VAE bottleneck vectors to CLIP embeddings.


## Freeze VAE
Freeze the trained fMRI autoencoder and encode each split with the latent mean vector.


In [ ]:
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def encode_tensor_mu(x_tensor, desc="encode fMRI"):
    loader = DataLoader(TensorDataset(x_tensor), batch_size=BATCH_SIZE, pin_memory=True)
    zs = []
    for (xb,) in tqdm(loader, desc=desc):
        h = model.encoder(xb.to(DEVICE, non_blocking=True))
        zs.append(model.mu(h).cpu())
    return torch.cat(zs)

z_train = encode_tensor_mu(x[train_idx], "encode train trials")
z_val = encode_tensor_mu(x[val_idx], "encode val trials")
z_test = encode_tensor_mu(x[test_idx], "encode test trials")
z_test_eval = encode_tensor_mu(x_test_eval, "encode test eval images")
yc_train = y_clip[train_idx]
yc_val = y_clip[val_idx]
yc_test = y_clip[test_idx]
yc_test_eval = y_clip_test_eval

test_eval_ids = test_eval_image_ids.tolist()

clip_mlp_train_loader = DataLoader(TensorDataset(z_train, yc_train), batch_size=BATCH_SIZE, shuffle=True, generator=g, pin_memory=True)
clip_mlp_train_eval_loader = DataLoader(TensorDataset(z_train, yc_train), batch_size=BATCH_SIZE, pin_memory=True)
clip_mlp_val_loader = DataLoader(TensorDataset(z_val, yc_val), batch_size=BATCH_SIZE, pin_memory=True)
clip_mlp_test_loader = DataLoader(TensorDataset(z_test_eval, yc_test_eval), batch_size=BATCH_SIZE, pin_memory=True)

print(f"latent={tuple(z_train.shape)} clip_dim={CLIP_DIM}")
print(f"test_eval_images={len(test_eval_ids)}")


## Bottleneck CLIP MLP Architecture
Define a supervised MLP that maps the frozen VAE bottleneck directly to normalized CLIP embeddings.


In [ ]:
class BottleneckCLIPMLP(nn.Module):
    def __init__(self, latent_dim, clip_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, clip_dim),
        )

    def forward(self, z):
        return self.net(z)

clip_mlp = BottleneckCLIPMLP(LATENT_DIM, CLIP_DIM, CLIP_MLP_HIDDEN_DIM, CLIP_MLP_DROPOUT).to(DEVICE)
clip_mlp_optimizer = torch.optim.AdamW(clip_mlp.parameters(), lr=CLIP_MLP_LR, weight_decay=1e-4)
print(f"clip_mlp_params={sum(p.numel() for p in clip_mlp.parameters()) / 1e6:.2f}M")


## Train Bottleneck CLIP MLP
Train direct supervised regression from VAE bottleneck vectors to paired CLIP targets.


In [ ]:
def bidirectional_clip_contrastive(pred_clip, true_clip, temperature=CLIP_MLP_TEMPERATURE):
    pred = F.normalize(pred_clip, dim=1)
    true = F.normalize(true_clip, dim=1)
    logits = pred @ true.T / temperature
    labels = torch.arange(len(pred), device=pred.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

def clip_mlp_loss(pred, true):
    mse = F.mse_loss(pred, true)
    cos = 1 - F.cosine_similarity(pred, true, dim=1).mean()
    contrastive = bidirectional_clip_contrastive(pred, true)
    loss = (
        CLIP_MLP_MSE_WEIGHT * mse
        + CLIP_MLP_COS_WEIGHT * cos
        + CLIP_MLP_CONTRASTIVE_WEIGHT * contrastive
    )
    return loss, mse, cos, contrastive

@torch.no_grad()
def predict_clip_from_bottleneck(z):
    clip_mlp.eval()
    z = z.to(DEVICE)
    return clip_mlp(z)

@torch.no_grad()
def evaluate_clip_mlp(loader):
    preds, trues = [], []
    clip_mlp.eval()
    for z, y in loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        pred = clip_mlp(z)
        preds.append(pred.cpu())
        trues.append(y.cpu())
    pred = torch.cat(preds)
    true = torch.cat(trues)
    mse = F.mse_loss(pred, true)
    cos = 1 - F.cosine_similarity(pred, true, dim=1).mean()
    contrastive = bidirectional_clip_contrastive(pred, true)
    total = (
        CLIP_MLP_MSE_WEIGHT * mse
        + CLIP_MLP_COS_WEIGHT * cos
        + CLIP_MLP_CONTRASTIVE_WEIGHT * contrastive
    )
    return {
        "loss": float(total),
        "mse": float(mse),
        "cos": float(cos),
        "contrastive": float(contrastive),
        "cos_sim": float(F.cosine_similarity(pred, true, dim=1).mean()),
    }

clip_mlp_history = []
best_clip_mlp_val = float("inf")
for epoch in range(1, CLIP_MLP_EPOCHS + 1):
    clip_mlp.train()
    totals = np.zeros(4)
    seen = 0
    pbar = tqdm(clip_mlp_train_loader, desc=f"clip mlp {epoch:03d}/{CLIP_MLP_EPOCHS}", leave=False)
    for z, y in pbar:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        pred = clip_mlp(z)
        loss, mse, cos, contrastive = clip_mlp_loss(pred, y)
        clip_mlp_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        clip_mlp_optimizer.step()
        totals += len(y) * np.array([loss.item(), mse.item(), cos.item(), contrastive.item()])
        seen += len(y)
        pbar.set_postfix(loss=loss.item(), mse=mse.item(), cos=cos.item(), ctr=contrastive.item())

    val_scores = evaluate_clip_mlp(clip_mlp_val_loader)
    row = {
        "epoch": epoch,
        "train_loss": totals[0] / seen,
        "train_mse": totals[1] / seen,
        "train_cos": totals[2] / seen,
        "train_contrastive": totals[3] / seen,
        **{f"val_{k}": v for k, v in val_scores.items()},
    }
    row["val_score"] = row["val_loss"]
    clip_mlp_history.append(row)
    tqdm.write(
        f"clip_mlp_epoch={epoch:03d} train={row['train_loss']:.4f} "
        f"val={row['val_loss']:.4f} val_mse={row['val_mse']:.4f} "
        f"val_cos={row['val_cos']:.4f} val_ctr={row['val_contrastive']:.4f}"
    )
    if row["val_score"] < best_clip_mlp_val:
        best_clip_mlp_val = row["val_score"]
        torch.save({
            "clip_mlp": clip_mlp.state_dict(),
            "clip_mean": clip_mean,
            "clip_std": clip_std,
            "model": "vae_bottleneck_clip_mlp",
        }, f"{OUTPUT_DIR}/best_clip_mlp.pt")

with open(f"{OUTPUT_DIR}/clip_mlp_history.json", "w") as f:
    json.dump(clip_mlp_history, f, indent=2)


## Evaluate Bottleneck CLIP MLP
Reload the best MLP and report train, validation, and test CLIP-translation losses.


In [ ]:
CLIP_MLP_HISTORY_PATH = f"{OUTPUT_DIR}/clip_mlp_history.json"
if "clip_mlp_history" not in globals() and os.path.exists(CLIP_MLP_HISTORY_PATH):
    with open(CLIP_MLP_HISTORY_PATH) as f:
        clip_mlp_history = json.load(f)

clip_mlp_ckpt = torch.load(f"{OUTPUT_DIR}/best_clip_mlp.pt", map_location=DEVICE)
clip_mlp.load_state_dict(clip_mlp_ckpt["clip_mlp"])
clip_mlp_train = evaluate_clip_mlp(clip_mlp_train_eval_loader)
clip_mlp_val = evaluate_clip_mlp(clip_mlp_val_loader)
clip_mlp_test = evaluate_clip_mlp(clip_mlp_test_loader)
print(f"clip_mlp_train_loss={clip_mlp_train['loss']:.6f} clip_mlp_train_mse={clip_mlp_train['mse']:.6f} clip_mlp_train_cos={clip_mlp_train['cos']:.6f} clip_mlp_train_contrastive={clip_mlp_train['contrastive']:.6f} cos_sim={clip_mlp_train['cos_sim']:.6f}")
print(f"clip_mlp_val_loss={clip_mlp_val['loss']:.6f} clip_mlp_val_mse={clip_mlp_val['mse']:.6f} clip_mlp_val_cos={clip_mlp_val['cos']:.6f} clip_mlp_val_contrastive={clip_mlp_val['contrastive']:.6f} cos_sim={clip_mlp_val['cos_sim']:.6f}")
print(f"clip_mlp_test_loss={clip_mlp_test['loss']:.6f} clip_mlp_test_mse={clip_mlp_test['mse']:.6f} clip_mlp_test_cos={clip_mlp_test['cos']:.6f} clip_mlp_test_contrastive={clip_mlp_test['contrastive']:.6f} cos_sim={clip_mlp_test['cos_sim']:.6f}")

if "clip_mlp_history" in globals() and len(clip_mlp_history):
    epochs = [h["epoch"] for h in clip_mlp_history]
    train_total = [h["train_loss"] for h in clip_mlp_history]
    val_total = [h.get("val_loss", CLIP_MLP_MSE_WEIGHT * h["val_mse"] + CLIP_MLP_COS_WEIGHT * h["val_cos"] + CLIP_MLP_CONTRASTIVE_WEIGHT * h.get("val_contrastive", 0.0)) for h in clip_mlp_history]
    plt.plot(epochs, train_total, label="train total")
    plt.plot(epochs, val_total, label="val total")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/clip_mlp_train_val_loss.png", dpi=180, bbox_inches="tight")
    plt.show()
    print(f"saved={OUTPUT_DIR}/clip_mlp_train_val_loss.png")
else:
    print("clip_mlp_history not found; skipping train/val loss plot")


# 3. Final BRAID v2 Evaluation
Evaluate frozen MLP predictions on the shared-1000 set with BRAID-style embedding and image metrics.


## Shared-1000 Image-Level Test Eval Set
Use NSD shared-1000 image IDs, with one averaged fMRI response, one CLIP embedding, and one ground-truth image per eval row.


In [ ]:
z_eval = z_test_eval
clip_mlp_eval_loader = clip_mlp_test_loader
eval_ids = test_eval_ids
print(f"shared1000_eval_images={len(eval_ids)}")
print(f"test_eval_betas={tuple(test_eval_betas.shape)} test_eval_clips={tuple(test_eval_clips.shape)}")
print("The final eval set is shared-1000 image-level: one averaged fMRI response, one CLIP embedding, and one image id per row.")


## Final Frozen Evaluation
Freeze the bottleneck CLIP MLP, collect limited test-set CLIP predictions, and score them against actual CLIP embeddings.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, HTML
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

clip_mlp.eval()
for p in clip_mlp.parameters():
    p.requires_grad_(False)

if "clip_mlp_eval_loader" not in globals():
    if "clip_mlp_test_loader" not in globals():
        raise NameError("clip_mlp_test_loader is missing. Run the Freeze VAE / bottleneck-cache cell before final evaluation.")
    clip_mlp_eval_loader = clip_mlp_test_loader
if "eval_ids" not in globals():
    if "test_eval_ids" in globals():
        eval_ids = test_eval_ids
    else:
        eval_ids = list(range(len(getattr(clip_mlp_eval_loader.dataset, "tensors", [clip_mlp_eval_loader.dataset])[0])))


@torch.no_grad()
def collect_clip_mlp_outputs(loader):
    preds, trues = [], []
    cm, cs = clip_mean.to(DEVICE), clip_std.to(DEVICE)
    for z, y in tqdm(loader, desc="collect final predictions"):
        pred_norm = predict_clip_from_bottleneck(z)
        true_norm = y.to(DEVICE)
        preds.append((pred_norm * cs + cm).cpu())
        trues.append((true_norm * cs + cm).cpu())
    return torch.cat(preds), torch.cat(trues)

def l2_normalize(a):
    return a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-8, None)

def mean_center(a):
    return a - a.mean(axis=0, keepdims=True)

def pairwise_sq_dists(a, b):
    return np.clip((a ** 2).sum(1, keepdims=True) + (b ** 2).sum(1, keepdims=True).T - 2 * a @ b.T, 0, None)

def mmd_gaussian(a, b, max_n=1000):
    rng = np.random.RandomState(SEED)
    if len(a) > max_n:
        a = a[rng.choice(len(a), max_n, replace=False)]
    if len(b) > max_n:
        b = b[rng.choice(len(b), max_n, replace=False)]
    z = np.concatenate([a, b])
    sub = z[rng.choice(len(z), size=min(len(z), 500), replace=False)]
    d2 = pairwise_sq_dists(sub, sub)
    gamma = 1.0 / (2 * np.median(d2[d2 > 0]))
    kxx = np.exp(-gamma * pairwise_sq_dists(a, a))
    kyy = np.exp(-gamma * pairwise_sq_dists(b, b))
    kxy = np.exp(-gamma * pairwise_sq_dists(a, b))
    m, n = len(a), len(b)
    return float((kxx.sum() - np.trace(kxx)) / (m * (m - 1)) + (kyy.sum() - np.trace(kyy)) / (n * (n - 1)) - 2 * kxy.mean())

def c2st_scores(pred_np, true_np, max_n=2000):
    rng = np.random.RandomState(SEED)
    n = min(len(pred_np), len(true_np), max_n // 2)
    pi = rng.choice(len(pred_np), n, replace=False)
    ti = rng.choice(len(true_np), n, replace=False)
    out = {}
    for name, p, t in [
        ("raw", pred_np[pi], true_np[ti]),
        ("l2_normalized", l2_normalize(pred_np[pi]), l2_normalize(true_np[ti])),
        ("mean_centered", mean_center(pred_np[pi]), mean_center(true_np[ti])),
    ]:
        x_cls = np.concatenate([p, t])
        y_cls = np.array([0] * len(p) + [1] * len(t))
        out[f"C2ST_LogReg_{name}"] = float(cross_val_score(LogisticRegression(max_iter=1000), x_cls, y_cls, cv=5).mean())
        out[f"C2ST_RBFSVM_{name}"] = float(cross_val_score(SVC(kernel="rbf"), x_cls, y_cls, cv=5).mean())
    return out

def retrieval_top1(pred, true, n_loops=30, n_samples=300):
    rng = np.random.RandomState(SEED)
    pred = F.normalize(pred.to(DEVICE), dim=1)
    true = F.normalize(true.to(DEVICE), dim=1)
    fwd, bwd = [], []
    for _ in range(n_loops):
        idx = rng.choice(len(true), size=min(n_samples, len(true)), replace=False)
        idx = torch.tensor(idx, device=DEVICE)
        labels = torch.arange(len(idx), device=DEVICE)
        fwd.append(((pred[idx] @ true[idx].T).argmax(1) == labels).float().mean().item())
        bwd.append(((true[idx] @ pred[idx].T).argmax(1) == labels).float().mean().item())
    fwd_ci = stats.norm.interval(0.95, loc=np.mean(fwd), scale=np.std(fwd) / np.sqrt(n_loops))
    bwd_ci = stats.norm.interval(0.95, loc=np.mean(bwd), scale=np.std(bwd) / np.sqrt(n_loops))
    return float(np.mean(fwd)), fwd_ci, float(np.mean(bwd)), bwd_ci

def metric_rows(items):
    return [{"Metric": name, "Value": f"{value:.6f}"} for name, value in items]

def display_report_table(rows, columns):
    df = pd.DataFrame(rows, columns=columns)
    html = df.to_html(index=False, escape=False, classes="eval-report-table", border=0)
    display(HTML("""
    <style>
      table.eval-report-table { border-collapse: collapse; margin: 6px 0 16px 0; min-width: 360px; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }
      table.eval-report-table th { background: #384152; color: #ffffff; font-weight: 700; text-align: left; padding: 8px 12px; border: 1px solid #5b6475; }
      table.eval-report-table td { padding: 8px 12px; border: 1px solid #5b6475; }
      table.eval-report-table tr:nth-child(even) td { background: rgba(127, 127, 127, 0.10); }
    </style>
    """ + html))

def show_eval_report(row, title, fwd_ci=None, bwd_ci=None):
    display(Markdown(f"### {title}"))
    display(Markdown("**Embedding + Retrieval**"))
    base_items = [("Embed MSE", row["EmbedMSE"]), ("Embed Cosine", row["EmbedCosine"]), ("MMD", row["MMD"]), ("Forward Retrieval", row["FwdRetrieval"]), ("Backward Retrieval", row["BwdRetrieval"])]
    display_report_table(metric_rows(base_items), ["Metric", "Value"])
    if fwd_ci is not None and bwd_ci is not None:
        print(f"Forward retrieval 95% CI: [{fwd_ci[0]:.4f}, {fwd_ci[1]:.4f}]")
        print(f"Backward retrieval 95% CI: [{bwd_ci[0]:.4f}, {bwd_ci[1]:.4f}]")
    display(Markdown("**Mixing Diagnostics**"))
    display_report_table(metric_rows([("Mix Silhouette", row["MixSilhouette"]), ("Mix Domain Accuracy", row["MixDomainAcc"])]), ["Metric", "Value"])
    c2st_rows = []
    for view in ["raw", "l2_normalized", "mean_centered"]:
        c2st_rows.append({"View": view, "LogReg": f"{row[f'C2ST_LogReg_{view}']:.6f}", "RBF-SVM": f"{row[f'C2ST_RBFSVM_{view}']:.6f}"})
    display_report_table(c2st_rows, ["View", "LogReg", "RBF-SVM"])
    image_keys = ["PixCorr", "SSIM", "AlexNet(2)", "AlexNet(5)", "InceptionV3", "CLIP", "EffNet-B", "SwAV"]
    if all(k in row for k in image_keys):
        display(Markdown("**Image Reconstruction Metrics**"))
        display_report_table(metric_rows([(k, row[k]) for k in image_keys]), ["Metric", "Value"])

pred_clip_raw, true_clip_raw = collect_clip_mlp_outputs(clip_mlp_eval_loader)
pred_np, true_np = pred_clip_raw.numpy(), true_clip_raw.numpy()
mix_n = min(len(pred_np), 1000)
mix_idx = np.random.RandomState(SEED).choice(len(pred_np), mix_n, replace=False)
mix_x = np.concatenate([pred_np[mix_idx], true_np[mix_idx]])
mix_y = np.array([0] * mix_n + [1] * mix_n)
fwd, fwd_ci, bwd, bwd_ci = retrieval_top1(pred_clip_raw, true_clip_raw)

final_eval = {
    "model": "vae_bottleneck_clip_mlp",
    "EmbedMSE": float(F.mse_loss(pred_clip_raw, true_clip_raw)),
    "EmbedCosine": float(F.cosine_similarity(pred_clip_raw, true_clip_raw, dim=1).mean()),
    "MixSilhouette": float(silhouette_score(mix_x, mix_y, metric="cosine")),
    "MixDomainAcc": float(cross_val_score(LogisticRegression(max_iter=1000), mix_x, mix_y, cv=5).mean()),
    "MMD": mmd_gaussian(pred_np, true_np),
    "FwdRetrieval": fwd,
    "BwdRetrieval": bwd,
}
final_eval.update(c2st_scores(pred_np, true_np))

eval_df = pd.DataFrame([final_eval]).set_index("model")
show_eval_report(final_eval, "Final Frozen Bottleneck CLIP MLP Eval", fwd_ci, bwd_ci)
eval_df.to_csv(f"{OUTPUT_DIR}/final_clip_mlp_eval.csv")
torch.save({"pred_clip_raw": pred_clip_raw, "true_clip_raw": true_clip_raw, "eval_ids": eval_ids}, f"{OUTPUT_DIR}/final_clip_predictions.pt")
print(f"saved={OUTPUT_DIR}/final_clip_mlp_eval.csv")


## Full Image Reconstruction Evals
Decode all shared-1000 predicted CLIP embeddings and compute the BRAID image-eval metrics.


In [ ]:
RUN_IMAGE_EVALS = True
IMAGE_DECODE_BATCH = 32
PREDICTIONS_PATH = f"{OUTPUT_DIR}/final_clip_predictions.pt"
FINAL_EVAL_CSV = f"{OUTPUT_DIR}/final_clip_mlp_eval.csv"
FULL_IMAGE_CACHE_PATH = FULL_IMAGE_CACHE_PATH
SHARED1000_CACHE_PATH = SHARED1000_CACHE_PATH
LOCAL_SHARED1000_CACHE_PATH = LOCAL_SHARED1000_CACHE_PATH

if RUN_IMAGE_EVALS and "pd" not in globals():
    import pandas as pd
if RUN_IMAGE_EVALS and "show_eval_report" not in globals():
    def show_eval_report(row, title, fwd_ci=None, bwd_ci=None):
        print(title)
        for k, v in row.items():
            if k != "model" and isinstance(v, (int, float, np.floating)):
                print(f"{k}: {float(v):.6f}")
if RUN_IMAGE_EVALS and "fwd_ci" not in globals():
    fwd_ci = None
if RUN_IMAGE_EVALS and "bwd_ci" not in globals():
    bwd_ci = None
if RUN_IMAGE_EVALS and "final_eval" not in globals():
    if os.path.exists(FINAL_EVAL_CSV):
        final_eval = pd.read_csv(FINAL_EVAL_CSV).iloc[0].to_dict()
        final_eval["model"] = final_eval.get("model", "vae_bottleneck_clip_mlp")
    else:
        final_eval = {"model": "vae_bottleneck_clip_mlp"}

if RUN_IMAGE_EVALS and "pred_clip_raw" not in globals():
    if not os.path.exists(PREDICTIONS_PATH):
        raise FileNotFoundError(
            f"Missing {PREDICTIONS_PATH}. Run the final frozen MLP evaluation cell before image eval."
        )
    prediction_payload = torch.load(PREDICTIONS_PATH, map_location="cpu")
    pred_clip_raw = prediction_payload["pred_clip_raw"]
    true_clip_raw = prediction_payload["true_clip_raw"]
    eval_ids = prediction_payload.get("eval_ids", list(range(len(pred_clip_raw))))
    pred_np, true_np = pred_clip_raw.numpy(), true_clip_raw.numpy()

if RUN_IMAGE_EVALS:
    if not os.path.exists(SHARED1000_CACHE_PATH) and not os.path.exists(LOCAL_SHARED1000_CACHE_PATH):
        raise FileNotFoundError(f"Missing shared-1000 cache: {SHARED1000_CACHE_PATH}")
    else:
        import sys, subprocess, scipy as sp

        def copy_with_progress(src, dst, desc, chunk_mb=64):
            total = os.path.getsize(src)
            tmp = f"{dst}.part"
            if os.path.exists(tmp):
                os.remove(tmp)
            with open(src, "rb") as fsrc, open(tmp, "wb") as fdst, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
                while True:
                    chunk = fsrc.read(chunk_mb * 1024 * 1024)
                    if not chunk:
                        break
                    fdst.write(chunk)
                    pbar.update(len(chunk))
            if os.path.getsize(tmp) != total:
                raise IOError(f"Incomplete copy: {tmp} has {os.path.getsize(tmp)} bytes, expected {total}")
            shutil.copystat(src, tmp)
            os.replace(tmp, dst)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers", "accelerate", "scikit-image"])
        from torchvision import transforms
        from torchvision.models.feature_extraction import create_feature_extractor
        from torchvision.models import alexnet, AlexNet_Weights, efficientnet_b1, EfficientNet_B1_Weights, inception_v3, Inception_V3_Weights
        from skimage.color import rgb2gray
        from skimage.metrics import structural_similarity as ssim_fn
        from diffusers import KandinskyV22Pipeline, KandinskyV22PriorPipeline

        def load_eval_images(eval_image_ids):
            eval_image_ids = list(map(int, eval_image_ids))
            if os.path.exists(FULL_IMAGE_CACHE_PATH):
                full_images_u8 = torch.load(FULL_IMAGE_CACHE_PATH, map_location="cpu")
                max_id = max(eval_image_ids)
                if max_id >= len(full_images_u8):
                    raise IndexError(f"FULL_IMAGE_CACHE_PATH has {len(full_images_u8)} images but eval image id {max_id} is requested")
                return full_images_u8[eval_image_ids].float().div(255)
            if os.path.exists(LOCAL_SHARED1000_CACHE_PATH):
                shared_images_u8 = torch.load(LOCAL_SHARED1000_CACHE_PATH, map_location="cpu")
            else:
                copy_with_progress(SHARED1000_CACHE_PATH, LOCAL_SHARED1000_CACHE_PATH, "copy shared-1000 cache")
                shared_images_u8 = torch.load(LOCAL_SHARED1000_CACHE_PATH, map_location="cpu")
            expected_shared_ids = list(map(int, sharedix[:len(eval_image_ids)])) if "sharedix" in globals() else list(range(len(shared_images_u8)))[:len(eval_image_ids)]
            if eval_image_ids == expected_shared_ids:
                return shared_images_u8[:len(eval_image_ids)].float().div(255)
            raise FileNotFoundError(
                "Need matched image rows. Either provide FULL_IMAGE_CACHE_PATH indexed by NSD image id, "
                "or use shared-1000 eval ids in the same order as shared1000_ground_truth_images_u8.pt. "
                f"Got {len(eval_image_ids)} eval ids and {len(shared_images_u8)} shared images, but ids/order do not match."
            )

        all_images = load_eval_images(eval_ids)
        print("all_images", tuple(all_images.shape))

        kandinsky_pbar = tqdm(total=3, desc="load Kandinsky", unit="stage")
        kandinsky_pbar.set_postfix_str("decoder")
        decoder = KandinskyV22Pipeline.from_pretrained("kandinsky-community/kandinsky-2-2-decoder", torch_dtype=torch.float16).to(DEVICE)
        kandinsky_pbar.update(1)
        kandinsky_pbar.set_postfix_str("prior")
        prior = KandinskyV22PriorPipeline.from_pretrained("kandinsky-community/kandinsky-2-2-prior", torch_dtype=torch.float16).to(DEVICE)
        kandinsky_pbar.update(1)
        kandinsky_pbar.set_postfix_str("negative embed")
        neg_embed = prior.get_zero_embed(1).to(DEVICE, torch.float16)
        kandinsky_pbar.update(1)
        kandinsky_pbar.close()

        def decode_predicted_images(batch_size=IMAGE_DECODE_BATCH):
            cached = f"{OUTPUT_DIR}/all_recons.pt"
            if os.path.exists(cached):
                print(f"loading cached reconstructions: {cached}")
                return torch.load(cached, map_location="cpu")
            embeds = pred_clip_raw.to(device=DEVICE, dtype=torch.float16)
            recons = []
            for start in tqdm(range(0, len(embeds), batch_size), desc=f"decoding {len(embeds)} images"):
                batch = embeds[start:start + batch_size]
                imgs = decoder(image_embeds=batch, negative_image_embeds=neg_embed.repeat(len(batch), 1), num_inference_steps=50, height=512, width=512).images
                recons.extend(transforms.ToTensor()(im) for im in imgs)
            all_recons = torch.stack(recons)
            torch.save(all_recons, cached)
            print(f"saved reconstructions: {cached} {tuple(all_recons.shape)}")
            return all_recons

        all_recons = decode_predicted_images()
        image_eval_n = min(len(all_recons), len(all_images))
        if len(all_recons) != len(all_images):
            print(f"warning: recon/image count mismatch; evaluating first {image_eval_n} pairs ({len(all_recons)} recons, {len(all_images)} images)")
        all_recons_eval = all_recons[:image_eval_n]
        all_images_eval = all_images[:image_eval_n]

        def eval_pixcorr(recons, images):
            resize = transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR)
            r = resize(images).reshape(len(images), -1).cpu().numpy()
            f = resize(recons).reshape(len(recons), -1).cpu().numpy()
            return float(np.mean([np.corrcoef(r[i], f[i])[0, 1] for i in range(len(r))]))

        def eval_ssim(recons, images):
            resize = transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR)
            img_gray = rgb2gray(resize(images).permute(0, 2, 3, 1).cpu().numpy())
            rec_gray = rgb2gray(resize(recons).permute(0, 2, 3, 1).cpu().numpy())
            scores = [ssim_fn(rec, im, data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False) for rec, im in zip(rec_gray, img_gray)]
            return float(np.mean(scores))

        @torch.no_grad()
        def two_way_identification(recons, images, feature_model, preprocess, feature_layer=None):
            preds = feature_model(torch.stack([preprocess(r) for r in recons]).to(DEVICE))
            reals = feature_model(torch.stack([preprocess(im) for im in images]).to(DEVICE))
            if feature_layer is not None:
                preds, reals = preds[feature_layer], reals[feature_layer]
            preds = preds.float().flatten(1).cpu().numpy()
            reals = reals.float().flatten(1).cpu().numpy()
            r = np.corrcoef(reals, preds)[:len(images), len(images):]
            success = r < np.diag(r)
            return float(np.mean(np.sum(success, 0)) / (len(images) - 1))

        alex_model = create_feature_extractor(alexnet(weights=AlexNet_Weights.IMAGENET1K_V1), return_nodes=["features.4", "features.11"]).to(DEVICE).eval().requires_grad_(False)
        alex_preprocess = transforms.Compose([transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        inception_model = create_feature_extractor(inception_v3(weights=Inception_V3_Weights.DEFAULT), return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)
        inception_preprocess = transforms.Compose([transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        eff_model = create_feature_extractor(efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT), return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)
        eff_preprocess = transforms.Compose([transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        swav_model = torch.hub.load("facebookresearch/swav:main", "resnet50")
        swav_model = create_feature_extractor(swav_model, return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)
        swav_preprocess = transforms.Compose([transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
        import clip as openai_clip
        clip_2way_model, _ = openai_clip.load("ViT-L/14", device=DEVICE)
        clip_2way_preprocess = transforms.Compose([transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])])

        def eval_effnet(recons, images):
            gt = eff_model(eff_preprocess(images).to(DEVICE))["avgpool"].reshape(len(images), -1).cpu().numpy()
            fk = eff_model(eff_preprocess(recons).to(DEVICE))["avgpool"].reshape(len(recons), -1).cpu().numpy()
            return float(np.mean([sp.spatial.distance.correlation(gt[i], fk[i]) for i in range(len(gt))]))

        def eval_swav(recons, images):
            gt = swav_model(swav_preprocess(images).to(DEVICE))["avgpool"].reshape(len(images), -1).cpu().numpy()
            fk = swav_model(swav_preprocess(recons).to(DEVICE))["avgpool"].reshape(len(recons), -1).cpu().numpy()
            return float(np.mean([sp.spatial.distance.correlation(gt[i], fk[i]) for i in range(len(gt))]))

        image_eval = dict(final_eval)
        image_eval.update({
            "PixCorr": eval_pixcorr(all_recons_eval, all_images_eval),
            "SSIM": eval_ssim(all_recons_eval, all_images_eval),
            "AlexNet(2)": two_way_identification(all_recons_eval, all_images_eval, alex_model, alex_preprocess, "features.4"),
            "AlexNet(5)": two_way_identification(all_recons_eval, all_images_eval, alex_model, alex_preprocess, "features.11"),
            "InceptionV3": two_way_identification(all_recons_eval, all_images_eval, inception_model, inception_preprocess, "avgpool"),
            "CLIP": two_way_identification(all_recons_eval, all_images_eval, clip_2way_model.encode_image, clip_2way_preprocess),
            "EffNet-B": eval_effnet(all_recons_eval, all_images_eval),
            "SwAV": eval_swav(all_recons_eval, all_images_eval),
        })
        image_eval_df = pd.DataFrame([image_eval]).set_index("model")
        show_eval_report(image_eval, "Final Full BRAID v2 Image Eval", fwd_ci, bwd_ci)
        image_eval_df.to_csv(f"{OUTPUT_DIR}/final_full_image_eval.csv")
        print(f"saved={OUTPUT_DIR}/final_full_image_eval.csv")


## Reconstruction Preview Grid
Display and save a large actual-vs-predicted image preview from cached reconstructions, without rerunning decoding or metrics.

In [ ]:
PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6
PREVIEW_GRID_PATH = f"{OUTPUT_DIR}/final_reconstruction_preview_grid.png"
LOCAL_SHARED1000_CACHE_PATH = LOCAL_SHARED1000_CACHE_PATH
SHARED1000_CACHE_PATH = SHARED1000_CACHE_PATH
FULL_IMAGE_CACHE_PATH = FULL_IMAGE_CACHE_PATH
RECONS_PATH = f"{OUTPUT_DIR}/all_recons.pt"
PREDICTIONS_PATH = f"{OUTPUT_DIR}/final_clip_predictions.pt"

assert os.path.exists(RECONS_PATH), f"missing reconstructions: {RECONS_PATH}"

if "all_recons" not in globals():
    all_recons = torch.load(RECONS_PATH, map_location="cpu")

if "all_images" not in globals():
    if "eval_ids" not in globals() and os.path.exists(PREDICTIONS_PATH):
        eval_ids = torch.load(PREDICTIONS_PATH, map_location="cpu").get("eval_ids", list(range(len(all_recons))))
    if os.path.exists(FULL_IMAGE_CACHE_PATH):
        full_images_u8 = torch.load(FULL_IMAGE_CACHE_PATH, map_location="cpu")
        all_images = full_images_u8[list(map(int, eval_ids))].float().div(255)
        del full_images_u8
    elif os.path.exists(LOCAL_SHARED1000_CACHE_PATH):
        all_images_u8 = torch.load(LOCAL_SHARED1000_CACHE_PATH, map_location="cpu")
        expected_shared_ids = list(map(int, sharedix[:len(eval_ids)])) if "sharedix" in globals() else list(range(len(all_images_u8)))[:len(eval_ids)]
        if list(map(int, eval_ids)) != expected_shared_ids:
            raise FileNotFoundError("Need matched image rows: provide FULL_IMAGE_CACHE_PATH or rerun with shared-1000 eval ids in shared cache order")
        all_images = all_images_u8[:len(eval_ids)].float().div(255)
        del all_images_u8
    else:
        raise FileNotFoundError(f"Missing image cache: {FULL_IMAGE_CACHE_PATH} or {LOCAL_SHARED1000_CACHE_PATH}")

if "eval_ids" not in globals() and os.path.exists(PREDICTIONS_PATH):
    eval_ids = torch.load(PREDICTIONS_PATH, map_location="cpu").get("eval_ids", list(range(len(all_recons))))

preview_pool_n = min(len(all_recons), len(all_images))
if len(all_recons) != len(all_images):
    print(f"warning: recon/image count mismatch; previewing from first {preview_pool_n} pairs ({len(all_recons)} recons, {len(all_images)} images)")
n_show = min(PREVIEW_GRID_N, preview_pool_n)
rng = np.random.RandomState(SEED)
show_idx = rng.choice(preview_pool_n, size=n_show, replace=False)
cols = min(PREVIEW_GRID_COLS, n_show)
rows = int(np.ceil(n_show / cols))

actual = F.interpolate(all_images[show_idx].float(), size=all_recons.shape[-2:], mode="bilinear", align_corners=False)
pred = all_recons[show_idx].float()
combined = torch.cat([actual, pred], dim=3).clamp(0, 1)

fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 3.0 * rows), constrained_layout=True)
axes = np.atleast_1d(axes).reshape(rows, cols)
for ax in axes.ravel():
    ax.axis("off")

for k, idx in enumerate(show_idx):
    ax = axes[k // cols, k % cols]
    ax.imshow(combined[k].permute(1, 2, 0))
    label = eval_ids[int(idx)] if "eval_ids" in globals() else int(idx)
    ax.set_title(f"eval id {label}\nactual | predicted", fontsize=9)
    ax.axis("off")

fig.suptitle("Actual NSD Images vs Predicted Reconstructions", fontsize=16)
plt.savefig(PREVIEW_GRID_PATH, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved={PREVIEW_GRID_PATH}")
